Session 2 RQ2 independent verification v1. Real baseline checks and oracle diagnostics; parser/fallback seam tests use explicitly synthetic responses. No B/C inference or scores are claimed.

In [ ]:
from pathlib import Path
import json, csv, hashlib, math, time, subprocess, sys, importlib.util
from collections import defaultdict, Counter
from datetime import datetime, timezone, date, timedelta
ROOT=Path.cwd()
OWN=ROOT / "llm4rec_s2_rq2"
INPUT=OWN / "inputs"
EVIDENCE=OWN / "evidence"
EVIDENCE.mkdir(exist_ok=True)
DATA=ROOT / json.loads((OWN / "data-path.json").read_text())["relative"]
protocol=json.loads((OWN / "baseline_seed0/protocol.json").read_text())
metrics=json.loads((OWN / "baseline_seed0/metrics.json").read_text())
contexts=[json.loads(l) for l in (OWN / "baseline_seed0/contexts.jsonl").read_text().splitlines()]
by_uid={c["user_id"]:c for c in contexts}
users=protocol["users"]
cutoff=protocol["cutoff"]
end=protocol["end_exclusive"]
def rows(p):
    with p.open(encoding="utf-8-sig", newline="") as f: yield from csv.DictReader(f)
def latest(p,key):
    out={}
    for r in rows(p):
        if r["timestamp"][:10]<cutoff and (r[key] not in out or r["timestamp"]>out[r[key]]["timestamp"]):out[r[key]]=r
    return out
assets=latest(DATA / "asset_information.csv","ISIN")
profiles=latest(DATA / "customer_information.csv","customerID")
priced={r["ISIN"] for r in rows(DATA / "close_prices.csv") if r["timestamp"][:10]<cutoff and float(r["closePrice"])>0}
assets={i:r for i,r in assets.items() if i in priced}
assert sorted(assets)==protocol["eligible_assets"]
assert cutoff=="2021-06-01" and end=="2021-07-01" and protocol["candidate_count"]==50 and not protocol["exclude_seen"]
source_hashes={n:hashlib.sha256((DATA/n).read_bytes()).hexdigest() for n in protocol["source_sha256"]}
assert source_hashes==protocol["source_sha256"]
payload={k:v for k,v in protocol.items() if k!="protocol_hash"}
assert hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()==protocol["protocol_hash"]
assert protocol["protocol_hash"].startswith("50efd4aeb415")
history=defaultdict(Counter); truth=defaultdict(set); seen=set(); excluded=Counter()
for r in rows(DATA / "transactions.csv"):
    rowkey=tuple(sorted(r.items()))
    if rowkey in seen:excluded["exact_duplicate_rows"]+=1;continue
    seen.add(rowkey)
    if int(r["transactionID"])<0:excluded["reconstructed_initial_position_rows"]+=1;continue
    if r["transactionType"]!="Buy" or r["ISIN"] not in assets:continue
    if r["timestamp"][:10]<cutoff:history[r["customerID"]][r["ISIN"]]+=1
    elif r["timestamp"][:10]<end:truth[r["customerID"]].add(r["ISIN"])
assert dict(excluded)==metrics["excluded_rows"]
eligible_users=[u for u in profiles if history.get(u)]
selected=sorted(eligible_users,key=lambda u:(hashlib.sha256(("42:"+u).encode()).hexdigest(),u))[:100]
assert selected==users and [c["user_id"] for c in contexts]==users
keys=("assetCategory","assetSubCategory","sector","industry","marketID")
features={i:sorted({f"{k}={r[k].strip().casefold()}" for k in keys if r.get(k,"").strip() not in ("","Not_Available","nan")}) for i,r in assets.items()}
pop=Counter(i for counts in history.values() for i in counts)
for c in contexts:
    uid=c["user_id"]
    assert [(a["id"],a["buy_count"]) for a in c["history"]]==sorted(history[uid].items(),key=lambda p:(-p[1],p[0]))
    profile=defaultdict(float)
    for i,n in history[uid].items():
        fs=features[i]
        for f in fs:profile[f]+=n/math.sqrt(len(fs))
    norm=math.sqrt(sum(x*x for x in profile.values()))
    scores={i:sum(profile[f]/math.sqrt(len(fs)) for f in fs)/norm if norm else 0.0 for i,fs in features.items()}
    if not any(scores.values()):scores={i:pop[i] for i in assets}
    candidates=sorted(scores,key=lambda i:(-scores[i],i))[:50]
    actual_ids=[a["id"] for a in c["candidate_assets"]]
    assert len(actual_ids)==len(set(actual_ids))==50
    assert all(scores[actual_ids[j]]+1e-14 >= scores[actual_ids[j+1]] for j in range(49))
    assert min(scores[i] for i in actual_ids)+1e-14 >= max(scores[i] for i in scores if i not in actual_ids)
    assert json.loads((OWN/"seeded-baseline-manifest.json").read_text())["seeded_starter_exact_repeat_parity"]
    assert c["eligible_asset_ids"]==sorted(assets)
    assert all(a["attributes"]==features[a["id"]] and a["metadata_timestamp"][:10]<cutoff for a in c["history"]+c["candidate_assets"])

predictions=defaultdict(list)
for r in rows(OWN / "baseline_seed0/predictions.csv"):
    key=(r["method"],r["user_id"])
    assert int(r["rank"])==len(predictions[key])+1
    predictions[key].append(r["ISIN"])
methods=sorted(metrics["methods"])
assert methods==["cbf","latest_buy","latest_buy_or_sell","popularity"]
assert set(predictions)=={(m,u) for m in methods for u in users}
def independent_score(ranking,target):
    discounts=[1.0/math.log2(r+1) for r in range(1,11)]
    denominator=sum(discounts[:min(10,len(target))])
    gain=sum(d for i,d in zip(ranking,discounts) if i in target)
    return gain/denominator if denominator else 0.0
per_user={(r["method"],r["user_id"]):r for r in rows(OWN/"baseline_seed0/per_user.csv")}
recomputed={}
for method in methods:
    values=[]; nonempty=[]
    for uid in users:
        ranking=predictions[(method,uid)]
        assert len(ranking)==len(set(ranking))==10 and set(ranking)<=set(assets)
        value=independent_score(ranking,truth[uid])
        assert abs(value-float(per_user[(method,uid)]["ndcg_at_10"]))<1e-14
        assert len(truth[uid])==int(per_user[(method,uid)]["relevant_count"])
        values.append(value)
        if truth[uid]:nonempty.append(value)
    actual={"users":len(values),"nonempty_truth_users":len(nonempty),"empty_truth_users":len(values)-len(nonempty),"ndcg_at_10":sum(values)/len(values),"ndcg_at_10_nonempty":sum(nonempty)/len(nonempty)}
    for k,v in actual.items():assert abs(v-metrics["methods"][method][k])<1e-14
    recomputed[method]=actual
assert sum(bool(truth[u]) for u in users)==11
reach=[]
for uid in users:
    candidate_ids=[a["id"] for a in by_uid[uid]["candidate_assets"]]
    reachable=truth[uid]&set(candidate_ids)
    oracle=sorted(reachable)+[i for i in candidate_ids if i not in reachable]
    reach.append({"user_id":uid,"truth_ids":sorted(truth[uid]),"truth_count":len(truth[uid]),"candidate_hit_ids":sorted(reachable),"reachable_count":len(reachable),"oracle_ndcg_at_10":independent_score(oracle[:10],truth[uid]),"history_assets":len(history[uid]),"buy_transactions":sum(history[uid].values())})
with (EVIDENCE/"target-reachability.jsonl").open("x") as f:
    for r in reach:f.write(json.dumps(r)+"\n")
diagnostic={"users":100,"nonempty_truth_users":11,"empty_truth_users":89,"nonempty_users_with_reachable_target":sum(bool(r["reachable_count"]) for r in reach),"target_asset_user_pairs":sum(r["truth_count"] for r in reach),"reachable_target_asset_user_pairs":sum(r["reachable_count"] for r in reach),"top50_oracle_ndcg_at_10_all":sum(r["oracle_ndcg_at_10"] for r in reach)/100,"top50_oracle_ndcg_at_10_nonempty":sum(r["oracle_ndcg_at_10"] for r in reach)/11,"history_assets_over_cap20_users":sum(r["history_assets"]>20 for r in reach)}
(EVIDENCE/"candidate-oracle.json").write_text(json.dumps(diagnostic,indent=2)+"\n")
(EVIDENCE/"baseline-independent-recomputation.json").write_text(json.dumps(recomputed,indent=2)+"\n")
print(json.dumps({"baseline":recomputed,"reachability":diagnostic},indent=2))


In [ ]:
spec=importlib.util.spec_from_file_location("shared",INPUT/"llm_rerank_original.py")
shared=importlib.util.module_from_spec(spec);spec.loader.exec_module(shared)
def guarded(text,n):
    try:return shared.parse_ranking(text,n),False
    except TypeError:return [],True
checks=[]
for text in ['{"ranking":null}','{"ranking":1}','{"ranking":1.2}','{"ranking":true}']:
    try:shared.parse_ranking(text,50)
    except TypeError:raised=True
    else:raised=False
    result,used=guarded(text,50)
    assert raised and used and result==[]
    checks.append({"input":text,"original_TypeError":True,"guarded_result":result,"invalid":True})
for text in ['{"ranking":[1,2,3,4,5,6,7,8,9,10]}','{"ranking":[true,0,51,2,2,1,"3"]}','bad','[]','{"ranking":"abc"}','{"ranking":{"a":1}}']:
    assert guarded(text,50)[0]==shared.parse_ranking(text,50)
    checks.append({"input":text,"original_behavior_preserved":True,"result":guarded(text,50)[0]})
(EVIDENCE/"parser-guard-checks.json").write_text(json.dumps(checks,indent=2)+"\n")
# Check both actual requested raw templates without running a model or fabricating a summary.
archive=[]
for ctx in contexts:
    shared.assert_no_future(ctx)
    projected={"cutoff":ctx["cutoff"],"history":[{k:a[k] for k in ["id","attributes","buy_count"]} for a in ctx["history"]],"candidate_assets":[{k:a[k] for k in ["id","attributes"]} for a in ctx["candidate_assets"]]}
    for purpose,actual,expected in [("B_rank",shared.rank_prompt(ctx,shared.history_block(ctx)),shared.rank_prompt(projected,shared.history_block(projected))),("C_summary",shared.summary_prompt(ctx,"C",False),shared.summary_prompt(projected,"C",False))]:
        assert actual==expected
        system,user=actual
        assert not any(field in system+user for field in ["riskLevel","investmentCapacity","customerType","lastQuestionnaireDate","assetName",'"truth"','"label"','"future"'])
        archive.append({"user_id":ctx["user_id"],"purpose":purpose,"system":system,"user":user,"executed_model":False,"template_projection_check":True})
with (EVIDENCE/"prepared-prompts-not-executed.jsonl").open("x") as f:
    for r in archive:f.write(json.dumps(r,ensure_ascii=False)+"\n")
# Explicitly SYNTHETIC seam-only check. These are never treated as model results.
synthetic=OWN/"checks/synthetic-fallback"
synthetic.mkdir(parents=True,exist_ok=False)
with (synthetic/"responses.jsonl").open("x") as f:
    for index,uid in enumerate(users):
        ids=[a["id"] for a in by_uid[uid]["candidate_assets"][:10]]
        if index==0:ids=None
        elif index==1:ids=1
        elif index==2:ids=[]
        elif index==3:ids=[ids[0]]*10
        f.write(json.dumps({"user_id":uid,"stage":"ranker","protocol_hash":protocol["protocol_hash"],"ranked_ids":ids})+"\n")
cmd=[sys.executable,str(INPUT/"starter.py"),"run","--data-dir",str(DATA),"--output-dir",str(synthetic/"scored"),"--stage","ranker","--responses",str(synthetic/"responses.jsonl"),"--on-invalid","fallback"]
t0=time.perf_counter();r=subprocess.run(cmd,env=dict(__import__("os").environ,PYTHONHASHSEED="0"),capture_output=True,text=True)
(synthetic/"console.log").write_text(r.stdout+r.stderr)
assert r.returncode==0,r.stdout+r.stderr
sm=json.loads((synthetic/"scored/metrics.json").read_text())
assert sm["protocol_hash"]==protocol["protocol_hash"] and len(sm["custom_failures"])==4
assert sm["methods"]["custom"]==metrics["methods"]["cbf"]
synthetic_predictions=defaultdict(list)
for r in rows(synthetic/"scored/predictions.csv"):synthetic_predictions[(r["method"],r["user_id"])].append(r["ISIN"])
assert all(synthetic_predictions[("custom",u)]==predictions[("cbf",u)] for u in users)
(EVIDENCE/"synthetic-fallback-check.json").write_text(json.dumps({"synthetic_only_not_model_results":True,"invalid_cases":["ranked_ids null","ranked_ids scalar","short list","duplicate ids"],"failures":sm["custom_failures"],"unchanged_starter_fallback_equals_CBF_for_all100":True,"command":cmd,"seconds":time.perf_counter()-t0},indent=2)+"\n")
# Informative actual baseline user, ready for a later student check. No personal verification claimed.
uid=max((u for u in users if truth[u]),key=lambda u:(max(independent_score(predictions[(m,u)],truth[u]) for m in methods)-min(independent_score(predictions[(m,u)],truth[u]) for m in methods),len(truth[u])))
target=truth[uid]
example={"user_id":uid,"selection_reason":"largest actual baseline nDCG spread among nonempty users", "student_personally_verified":False,"scope":"baseline evidence only; B/C unavailable", "truth_ids":sorted(target),"cutoff":cutoff,"end_exclusive":end,"candidate_ids":[a["id"] for a in by_uid[uid]["candidate_assets"]],"history":by_uid[uid]["history"],"ideal_dcg_at_10":sum(1/math.log2(i+2) for i in range(min(10,len(target)))),"methods":{}}
for m in methods:
    ranking=predictions[(m,uid)]
    example["methods"][m]={"ranking":ranking,"contributions":[{"rank":i+1,"id":a,"relevant":a in target,"gain":1/math.log2(i+2) if a in target else 0.0} for i,a in enumerate(ranking)],"ndcg_at_10":independent_score(ranking,target)}
(EVIDENCE/"student-check-example.json").write_text(json.dumps(example,indent=2)+"\n")
verification={"status":"baseline_and_preparation_checks_passed_BC_not_executed", "performed_by":"AI agent executed checks; no student personal verification claimed", "input_hashes":json.loads((OWN/"input-hashes.json").read_text()),"protocol_hash":protocol["protocol_hash"],"starter_tests":58,"checks":{"source_data_hashes":True,"protocol_digest":True,"deterministic_warm100_selection_independent":True,"candidate_top50_independent_scores_with_float_tie_tolerance_1e14":True,"exact_candidate_order_seeded_starter_rerun_parity":True,"precutoff_history_and_metadata":True,"baseline_unique_eligible_ten":True,"baseline_saved_scores_independently_recomputed":True,"shared_null_scalar_TypeError_and_narrow_guard":True,"valid_parser_behavior_unchanged":True,"starter_synthetic_invalid_fallback":True,"prepared_B_rank_and_C_summary_templates_no_name_profile_future_fields":True},"pending":["real 3-user smoke","100-user B and C inference","actual C-rank prompt audit","actual token counts and generation timings","saved B/C scoring with unchanged starter and fallback","independent B/C recomputation and paired differences"],"reachability":diagnostic}
(EVIDENCE/"verification.json").write_text(json.dumps(verification,indent=2)+"\n")
network=json.loads((OWN/"network-diagnostic.json").read_text())
summary={"status":"incomplete_network_blocked_no_model_inference", "rq":"Does an LLM-generated transaction-history summary improve future-purchase nDCG@10 over fixed raw history under the identical shared LLM reranker?", "protocol_hash":protocol["protocol_hash"],"model_requested":{"id":"Qwen/Qwen3-1.7B","revision":"70d244cc86ccca08cf5af4e1e306ecf908b1ad5e","torch_preferred":"2.11.0","transformers_preferred":"4.57.6","dtype":"float32","seed":0,"thinking":False,"do_sample":False,"summary_max_new_tokens":120,"rank_max_new_tokens":60},"baseline":metrics["methods"],"B":{"status":"not_executed","measured_metrics":None},"C":{"status":"not_executed","measured_metrics":None},"paired_C_minus_B":None,"baselines_timing":json.loads((OWN/"seeded-baseline-manifest.json").read_text())["runs"][0],"network_diagnostic":network,"preflight":json.loads((OWN/"preflight.json").read_text()),"determinism":json.loads((OWN/"seeded-baseline-manifest.json").read_text()),"GPU_allocations":0,"generation_calls":0,"reachability":diagnostic,"conclusion":"RQ2 remains unanswered: no B/C model outputs exist. Baseline scores are genuinely executed, independently recomputed outputs from this run.","student_check":"student-check-example.json contains actual baseline evidence; no personal verification claimed"}
(EVIDENCE/"results-summary.json").write_text(json.dumps(summary,indent=2)+"\n")
print(json.dumps({"verification_status":verification["status"],"summary_status":summary["status"],"student_check_user":uid},indent=2))
